# 02 — Silver (SCD1) — Dynamic Tables

Camada Silver com:
- **Dynamic Tables** (equivalente a Delta Tables no Snowflake — processam apenas o incremento)
- **SCD Tipo 1** — sobrescreve com a versão mais recente
- **Deduplicação** via ROW_NUMBER na chave composta
- **Rejeição** de registros com chave NULL ou duplicados → tabela de rejeitados
- **Campos de auditoria:** `_DATA_INCLUSAO`

### Bronze vs Silver
| Camada | Tipo Snowflake | Comportamento |
|--------|---------------|---------------|
| Bronze | Tabela estática | Append-only (histórico bruto) |
| Silver | **Dynamic Table** | Refresh automático incremental (delta) |

*Co-authored with CoCo*

In [ ]:
%%sql -r ctx_silver
USE WAREHOUSE YDS_WH;
USE SCHEMA YDS_DB.SILVER;

## Tabela de Rejeição

Registros rejeitados por:
1. **Chave NULL** — MATCH_ID ou LOCATION_X/Y são NULL (impossível identificar o arremesso)
2. **Duplicados** — mesma chave composta aparece mais de uma vez
3. **NULLs críticos** — colunas essenciais para análise estão NULL

In [ ]:
%%sql -r create_rejected
-- Tabela de rejeição (estática, append-only)
CREATE TABLE IF NOT EXISTS YDS_DB.RAW.SHOTS_REJECTED (
    -- Dados originais
    MATCH_EVENT_ID        FLOAT,
    LOCATION_X            FLOAT,
    LOCATION_Y            FLOAT,
    REMAINING_MIN         FLOAT,
    POWER_OF_SHOT         FLOAT,
    KNOCKOUT_MATCH        FLOAT,
    GAME_SEASON           VARCHAR(10),
    REMAINING_SEC         FLOAT,
    DISTANCE_OF_SHOT      FLOAT,
    IS_GOAL               FLOAT,
    AREA_OF_SHOT          VARCHAR(50),
    SHOT_BASICS           VARCHAR(50),
    RANGE_OF_SHOT         VARCHAR(50),
    TEAM_NAME             VARCHAR(100),
    DATE_OF_GAME          DATE,
    HOME_AWAY             VARCHAR(50),
    SHOT_ID_NUMBER        FLOAT,
    LATITUDE              FLOAT,
    LONGITUDE             FLOAT,
    TYPE_OF_SHOT          VARCHAR(50),
    TYPE_OF_COMBINED_SHOT VARCHAR(50),
    MATCH_ID              INTEGER,
    TEAM_ID               BIGINT,
    REMAINING_MIN_2       FLOAT,
    POWER_OF_SHOT_2       FLOAT,
    KNOCKOUT_MATCH_2      FLOAT,
    REMAINING_SEC_2       FLOAT,
    DISTANCE_OF_SHOT_2    FLOAT,
    -- Metadados de rejeição
    _MOTIVO_REJEICAO      VARCHAR(200),
    _COLUNAS_PROBLEMA     VARCHAR(500),
    _REJEITADO_EM         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    _SOURCE_BATCH_ID      VARCHAR(50)
);

In [ ]:
%%sql -r reject_null_keys
-- Inserir registros com CHAVE NULL (MATCH_ID ou coordenadas são obrigatórios)
INSERT INTO YDS_DB.RAW.SHOTS_REJECTED
SELECT 
    MATCH_EVENT_ID, LOCATION_X, LOCATION_Y, REMAINING_MIN, POWER_OF_SHOT,
    KNOCKOUT_MATCH, GAME_SEASON, REMAINING_SEC, DISTANCE_OF_SHOT, IS_GOAL,
    AREA_OF_SHOT, SHOT_BASICS, RANGE_OF_SHOT, TEAM_NAME, DATE_OF_GAME,
    HOME_AWAY, SHOT_ID_NUMBER, LATITUDE, LONGITUDE, TYPE_OF_SHOT,
    TYPE_OF_COMBINED_SHOT, MATCH_ID, TEAM_ID, REMAINING_MIN_2,
    POWER_OF_SHOT_2, KNOCKOUT_MATCH_2, REMAINING_SEC_2, DISTANCE_OF_SHOT_2,
    'CHAVE NULL' AS _MOTIVO_REJEICAO,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        CASE WHEN MATCH_ID IS NULL THEN 'MATCH_ID' END,
        CASE WHEN LOCATION_X IS NULL THEN 'LOCATION_X' END,
        CASE WHEN LOCATION_Y IS NULL THEN 'LOCATION_Y' END,
        CASE WHEN REMAINING_MIN IS NULL THEN 'REMAINING_MIN' END,
        CASE WHEN REMAINING_SEC IS NULL THEN 'REMAINING_SEC' END
    ), ', ') AS _COLUNAS_PROBLEMA,
    CURRENT_TIMESTAMP() AS _REJEITADO_EM,
    _BATCH_ID AS _SOURCE_BATCH_ID
FROM YDS_DB.RAW.YDS_SHOTS_BRONZE
WHERE MATCH_ID IS NULL 
   OR LOCATION_X IS NULL 
   OR LOCATION_Y IS NULL
   OR REMAINING_MIN IS NULL
   OR REMAINING_SEC IS NULL;

In [ ]:
%%sql -r reject_null_cols
-- Inserir registros com NULLs em colunas críticas para análise
-- (exceto IS_GOAL que é target ML)
INSERT INTO YDS_DB.RAW.SHOTS_REJECTED
SELECT 
    MATCH_EVENT_ID, LOCATION_X, LOCATION_Y, REMAINING_MIN, POWER_OF_SHOT,
    KNOCKOUT_MATCH, GAME_SEASON, REMAINING_SEC, DISTANCE_OF_SHOT, IS_GOAL,
    AREA_OF_SHOT, SHOT_BASICS, RANGE_OF_SHOT, TEAM_NAME, DATE_OF_GAME,
    HOME_AWAY, SHOT_ID_NUMBER, LATITUDE, LONGITUDE, TYPE_OF_SHOT,
    TYPE_OF_COMBINED_SHOT, MATCH_ID, TEAM_ID, REMAINING_MIN_2,
    POWER_OF_SHOT_2, KNOCKOUT_MATCH_2, REMAINING_SEC_2, DISTANCE_OF_SHOT_2,
    'NULL EM COLUNA CRITICA' AS _MOTIVO_REJEICAO,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        CASE WHEN GAME_SEASON IS NULL THEN 'GAME_SEASON' END,
        CASE WHEN AREA_OF_SHOT IS NULL THEN 'AREA_OF_SHOT' END,
        CASE WHEN SHOT_BASICS IS NULL THEN 'SHOT_BASICS' END,
        CASE WHEN RANGE_OF_SHOT IS NULL THEN 'RANGE_OF_SHOT' END,
        CASE WHEN DISTANCE_OF_SHOT IS NULL THEN 'DISTANCE_OF_SHOT' END,
        CASE WHEN POWER_OF_SHOT IS NULL THEN 'POWER_OF_SHOT' END,
        CASE WHEN HOME_AWAY IS NULL THEN 'HOME_AWAY' END,
        CASE WHEN KNOCKOUT_MATCH IS NULL THEN 'KNOCKOUT_MATCH' END
    ), ', ') AS _COLUNAS_PROBLEMA,
    CURRENT_TIMESTAMP() AS _REJEITADO_EM,
    _BATCH_ID AS _SOURCE_BATCH_ID
FROM YDS_DB.RAW.YDS_SHOTS_BRONZE
WHERE MATCH_ID IS NOT NULL AND LOCATION_X IS NOT NULL AND LOCATION_Y IS NOT NULL
  AND REMAINING_MIN IS NOT NULL AND REMAINING_SEC IS NOT NULL
  AND IS_GOAL IS NOT NULL
  AND (
    GAME_SEASON IS NULL OR AREA_OF_SHOT IS NULL OR SHOT_BASICS IS NULL OR
    RANGE_OF_SHOT IS NULL OR DISTANCE_OF_SHOT IS NULL OR POWER_OF_SHOT IS NULL OR
    HOME_AWAY IS NULL OR KNOCKOUT_MATCH IS NULL
  );

In [ ]:
%%sql -r reject_dups
-- Inserir duplicados na tabela de rejeição
-- Chave composta: MATCH_ID + LOCATION_X + LOCATION_Y + REMAINING_MIN + REMAINING_SEC
-- Mantém o primeiro registro (menor SHOT_ID_NUMBER), rejeita os demais
INSERT INTO YDS_DB.RAW.SHOTS_REJECTED
WITH ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY MATCH_ID, COALESCE(SHOT_ID_NUMBER, 0), LOCATION_X, LOCATION_Y, REMAINING_MIN, REMAINING_SEC
            ORDER BY COALESCE(MATCH_EVENT_ID, 999999), DATE_OF_GAME DESC
        ) AS RN
    FROM YDS_DB.RAW.YDS_SHOTS_BRONZE
    WHERE MATCH_ID IS NOT NULL AND LOCATION_X IS NOT NULL AND LOCATION_Y IS NOT NULL
      AND REMAINING_MIN IS NOT NULL AND REMAINING_SEC IS NOT NULL
      AND IS_GOAL IS NOT NULL
      AND GAME_SEASON IS NOT NULL AND AREA_OF_SHOT IS NOT NULL AND SHOT_BASICS IS NOT NULL
      AND RANGE_OF_SHOT IS NOT NULL AND DISTANCE_OF_SHOT IS NOT NULL
      AND POWER_OF_SHOT IS NOT NULL AND HOME_AWAY IS NOT NULL AND KNOCKOUT_MATCH IS NOT NULL
)
SELECT
    MATCH_EVENT_ID, LOCATION_X, LOCATION_Y, REMAINING_MIN, POWER_OF_SHOT,
    KNOCKOUT_MATCH, GAME_SEASON, REMAINING_SEC, DISTANCE_OF_SHOT, IS_GOAL,
    AREA_OF_SHOT, SHOT_BASICS, RANGE_OF_SHOT, TEAM_NAME, DATE_OF_GAME,
    HOME_AWAY, SHOT_ID_NUMBER, LATITUDE, LONGITUDE, TYPE_OF_SHOT,
    TYPE_OF_COMBINED_SHOT, MATCH_ID, TEAM_ID, REMAINING_MIN_2,
    POWER_OF_SHOT_2, KNOCKOUT_MATCH_2, REMAINING_SEC_2, DISTANCE_OF_SHOT_2,
    'REGISTRO DUPLICADO' AS _MOTIVO_REJEICAO,
    'Chave: MATCH_ID + SHOT_ID + LOC_X + LOC_Y + MIN + SEC' AS _COLUNAS_PROBLEMA,
    CURRENT_TIMESTAMP() AS _REJEITADO_EM,
    _BATCH_ID AS _SOURCE_BATCH_ID
FROM ranked
WHERE RN > 1;

In [ ]:
%%sql -r rejected_summary
-- Resumo da tabela de rejeição
SELECT _MOTIVO_REJEICAO, COUNT(*) AS REGISTROS
FROM YDS_DB.RAW.SHOTS_REJECTED
GROUP BY _MOTIVO_REJEICAO
ORDER BY REGISTROS DESC;

## Dynamic Tables Silver (SCD1)

As Dynamic Tables abaixo:
- Atualizam automaticamente quando novos dados entram na Bronze (TARGET_LAG = 1 hora)
- Processam apenas o delta (incremento) a cada refresh
- Aplicam deduplicação e limpeza automaticamente
- Equivalem a "Delta Tables" de outras plataformas

In [ ]:
%%sql -r drop_silver_shots
-- Limpar objetos existentes para recriar
DROP DYNAMIC TABLE IF EXISTS YDS_DB.SILVER.SHOTS_CLEANED;
DROP TABLE IF EXISTS YDS_DB.SILVER.SHOTS_CLEANED;

In [ ]:
%%sql -r create_silver_shots
-- Dynamic Table Silver: Arremessos limpos, deduplicados, SCD1
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.SILVER.SHOTS_CLEANED
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
WITH deduplicada AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY MATCH_ID, COALESCE(SHOT_ID_NUMBER, 0), LOCATION_X, LOCATION_Y, REMAINING_MIN, REMAINING_SEC
            ORDER BY COALESCE(MATCH_EVENT_ID, 999999), DATE_OF_GAME DESC
        ) AS RN
    FROM YDS_DB.RAW.YDS_SHOTS_BRONZE
    WHERE -- Exclui chaves NULL
          MATCH_ID IS NOT NULL AND LOCATION_X IS NOT NULL AND LOCATION_Y IS NOT NULL
      AND REMAINING_MIN IS NOT NULL AND REMAINING_SEC IS NOT NULL
      -- Exclui NULLs críticos
      AND GAME_SEASON IS NOT NULL AND AREA_OF_SHOT IS NOT NULL AND SHOT_BASICS IS NOT NULL
      AND RANGE_OF_SHOT IS NOT NULL AND DISTANCE_OF_SHOT IS NOT NULL
      AND POWER_OF_SHOT IS NOT NULL AND HOME_AWAY IS NOT NULL AND KNOCKOUT_MATCH IS NOT NULL
      -- Exclui dados para ML (IS_GOAL NULL)
      AND IS_GOAL IS NOT NULL
)
SELECT
    -- Chave natural
    MATCH_EVENT_ID,
    MATCH_ID,
    SHOT_ID_NUMBER,

    -- Atributos do jogo
    DATE_OF_GAME,
    GAME_SEASON,
    HOME_AWAY,
    CASE WHEN HOME_AWAY ILIKE '%@%' THEN 'Away' ELSE 'Home' END AS GAME_LOCATION,

    -- Localização do arremesso
    LOCATION_X,
    LOCATION_Y,
    AREA_OF_SHOT,
    SHOT_BASICS,
    RANGE_OF_SHOT,
    DISTANCE_OF_SHOT,

    -- Contexto temporal
    REMAINING_MIN,
    REMAINING_SEC,
    (REMAINING_MIN * 60) + REMAINING_SEC AS TOTAL_REMAINING_SEC,
    POWER_OF_SHOT,
    KNOCKOUT_MATCH AS IS_CUP_MATCH,

    -- Tipo de arremesso
    TYPE_OF_SHOT,
    TYPE_OF_COMBINED_SHOT,
    COALESCE(TYPE_OF_SHOT, TYPE_OF_COMBINED_SHOT) AS SHOT_TYPE_FINAL,

    -- Resultado
    IS_GOAL,
    CASE WHEN IS_GOAL = 1 THEN 'MADE' ELSE 'MISSED' END AS SHOT_RESULT,

    -- Geolocalização
    LATITUDE,
    LONGITUDE,

    -- Features processadas
    REMAINING_MIN_2,
    POWER_OF_SHOT_2,
    KNOCKOUT_MATCH_2,
    REMAINING_SEC_2,
    DISTANCE_OF_SHOT_2,

    -- Auditoria SCD1
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO

FROM deduplicada
WHERE RN = 1;

In [ ]:
%%sql -r create_silver_ml
-- Dynamic Table Silver: Arremessos limpos, deduplicados, SCD1
-- Tratamentos: dedup, flags de contexto, renomeação de features ML
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.SILVER.SHOTS_CLEANED
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
WITH deduplicada AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY MATCH_ID, COALESCE(SHOT_ID_NUMBER, 0), LOCATION_X, LOCATION_Y, REMAINING_MIN, REMAINING_SEC
            ORDER BY COALESCE(MATCH_EVENT_ID, 999999), DATE_OF_GAME DESC
        ) AS RN
    FROM YDS_DB.RAW.YDS_SHOTS_BRONZE
    WHERE -- Exclui chaves NULL
          MATCH_ID IS NOT NULL AND LOCATION_X IS NOT NULL AND LOCATION_Y IS NOT NULL
      AND REMAINING_MIN IS NOT NULL AND REMAINING_SEC IS NOT NULL
      -- Exclui NULLs críticos
      AND GAME_SEASON IS NOT NULL AND AREA_OF_SHOT IS NOT NULL AND SHOT_BASICS IS NOT NULL
      AND RANGE_OF_SHOT IS NOT NULL AND DISTANCE_OF_SHOT IS NOT NULL
      AND POWER_OF_SHOT IS NOT NULL AND HOME_AWAY IS NOT NULL AND KNOCKOUT_MATCH IS NOT NULL
      -- Exclui dados para ML (IS_GOAL NULL)
      AND IS_GOAL IS NOT NULL
)
SELECT
    -- Chave natural
    MATCH_EVENT_ID,
    MATCH_ID,
    SHOT_ID_NUMBER,

    -- Atributos do jogo
    DATE_OF_GAME,
    GAME_SEASON,
    HOME_AWAY,
    CASE WHEN HOME_AWAY ILIKE '%@%' THEN 'Away' ELSE 'Home' END AS GAME_LOCATION,
    CASE 
        WHEN HOME_AWAY ILIKE 'MANU @ %' THEN REPLACE(HOME_AWAY, 'MANU @ ', '')
        WHEN HOME_AWAY ILIKE 'MANU vs. %' THEN REPLACE(HOME_AWAY, 'MANU vs. ', '')
        ELSE HOME_AWAY
    END AS ADVERSARIO,

    -- Localização do arremesso
    LOCATION_X,
    LOCATION_Y,
    AREA_OF_SHOT,
    SHOT_BASICS,
    RANGE_OF_SHOT,
    DISTANCE_OF_SHOT,

    -- Contexto temporal
    REMAINING_MIN,
    REMAINING_SEC,
    (REMAINING_MIN * 60) + REMAINING_SEC AS TOTAL_REMAINING_SEC,
    POWER_OF_SHOT AS PERIODO,
    KNOCKOUT_MATCH AS IS_PLAYOFF,

    -- Tipo de arremesso
    TYPE_OF_SHOT,
    TYPE_OF_COMBINED_SHOT,
    COALESCE(TYPE_OF_SHOT, TYPE_OF_COMBINED_SHOT) AS SHOT_TYPE_FINAL,

    -- Resultado
    IS_GOAL,
    CASE WHEN IS_GOAL = 1 THEN 'MADE' ELSE 'MISSED' END AS SHOT_RESULT,

    -- Flags de contexto
    CASE WHEN DISTANCE_OF_SHOT > 47 THEN TRUE ELSE FALSE END AS IS_BACKCOURT_SHOT,
    CASE WHEN (REMAINING_MIN * 60) + REMAINING_SEC <= 120 THEN TRUE ELSE FALSE END AS IS_CLUTCH,
    CASE 
        WHEN DISTANCE_OF_SHOT <= 8 THEN 'Paint'
        WHEN DISTANCE_OF_SHOT <= 16 THEN 'Mid-Range Curto'
        WHEN DISTANCE_OF_SHOT <= 24 THEN 'Mid-Range Longo'
        ELSE 'Longa Distância'
    END AS SHOT_ZONE_CUSTOM,

    -- Geolocalização
    LATITUDE,
    LONGITUDE,

    -- Features ML (colunas _2 renomeadas para clareza)
    REMAINING_MIN_2 AS _FEAT_REMAINING_MIN,
    POWER_OF_SHOT_2 AS _FEAT_POWER,
    KNOCKOUT_MATCH_2 AS _FEAT_KNOCKOUT,
    REMAINING_SEC_2 AS _FEAT_REMAINING_SEC,
    DISTANCE_OF_SHOT_2 AS _FEAT_DISTANCE,

    -- Auditoria SCD1
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO

FROM deduplicada
WHERE RN = 1;

In [ ]:
%%sql -r create_silver_gamelog
-- Dynamic Table Silver: Gamelog enriquecido
DROP DYNAMIC TABLE IF EXISTS YDS_DB.SILVER.KOBE_GAMELOG;
DROP TABLE IF EXISTS YDS_DB.SILVER.KOBE_GAMELOG;

CREATE OR REPLACE DYNAMIC TABLE YDS_DB.SILVER.KOBE_GAMELOG
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    GAME_ID,
    DATE,
    SEASON,
    GAME_TYPE,
    CASE WHEN GAME_TYPE = 'PO' THEN 'Playoff' ELSE 'Regular Season' END AS GAME_TYPE_FULL,
    OPPONENT,
    VENUE,
    KOBE_POINTS,
    LAL_SCORE,
    OPP_SCORE,
    LAL_SCORE - OPP_SCORE AS POINT_DIFF,
    RESULT,
    CASE WHEN RESULT = 'W' THEN 1 ELSE 0 END AS IS_WIN,
    CASE WHEN ABS(LAL_SCORE - OPP_SCORE) >= 15 THEN 1 ELSE 0 END AS IS_BLOWOUT,
    CASE WHEN ABS(LAL_SCORE - OPP_SCORE) <= 5 THEN 1 ELSE 0 END AS IS_CLOSE_GAME,
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO
FROM YDS_DB.RAW.KOBE_GAMELOG_BRONZE;

In [ ]:
%%sql -r create_silver_seasons
-- Dynamic Table Silver: Season Summary enriquecido
DROP DYNAMIC TABLE IF EXISTS YDS_DB.SILVER.KOBE_SEASONS;
DROP TABLE IF EXISTS YDS_DB.SILVER.KOBE_SEASONS;

CREATE OR REPLACE DYNAMIC TABLE YDS_DB.SILVER.KOBE_SEASONS
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    SEASON,
    GAMES,
    PPG,
    WINS,
    LOSSES,
    WINS + LOSSES AS TOTAL_GAMES_TEAM,
    ROUND(WINS / NULLIF(WINS + LOSSES, 0) * 100, 1) AS WIN_PCT,
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO
FROM YDS_DB.RAW.KOBE_SEASON_SUMMARY_BRONZE;

## Validação

In [ ]:
%%sql -r silver_validation
-- Validar todas as Dynamic Tables Silver
SELECT 'SILVER.SHOTS_CLEANED' AS TABELA, COUNT(*) AS LINHAS FROM YDS_DB.SILVER.SHOTS_CLEANED
UNION ALL SELECT 'SILVER.SHOTS_ML_PREDICT', COUNT(*) FROM YDS_DB.SILVER.SHOTS_ML_PREDICT
UNION ALL SELECT 'SILVER.KOBE_GAMELOG', COUNT(*) FROM YDS_DB.SILVER.KOBE_GAMELOG
UNION ALL SELECT 'SILVER.KOBE_SEASONS', COUNT(*) FROM YDS_DB.SILVER.KOBE_SEASONS
UNION ALL SELECT 'RAW.SHOTS_REJECTED', COUNT(*) FROM YDS_DB.RAW.SHOTS_REJECTED;

In [ ]:
%%sql -r dedup_validation
-- Confirmar zero duplicados na Silver
SELECT
    COUNT(*) AS TOTAL,
    COUNT(DISTINCT MATCH_ID || '-' || COALESCE(SHOT_ID_NUMBER::VARCHAR,'0') || '-' || LOCATION_X || '-' || LOCATION_Y || '-' || REMAINING_MIN || '-' || REMAINING_SEC) AS CHAVES_UNICAS
FROM YDS_DB.SILVER.SHOTS_CLEANED;

In [ ]:
%%sql -r dt_status
-- Verificar Dynamic Tables status
SHOW DYNAMIC TABLES IN SCHEMA YDS_DB.SILVER;

## Dynamic Table: ARENAS (SCD1)

Transformação da `ARENAS_BRONZE` para Silver:
- Padronização de nomes
- Validação de datas (VALID_FROM < VALID_TO)
- Deduplicação por chave (TEAM_ABBR + VALID_FROM)
- Campo `_DATA_INCLUSAO` para auditoria

In [ ]:
%%sql -r create_silver_arenas
-- Dynamic Table Silver: Arenas com SCD1
DROP DYNAMIC TABLE IF EXISTS YDS_DB.SILVER.ARENAS;
DROP TABLE IF EXISTS YDS_DB.SILVER.ARENAS;

CREATE OR REPLACE DYNAMIC TABLE YDS_DB.SILVER.ARENAS
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
WITH dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY TEAM_ABBR, VALID_FROM
            ORDER BY _LOADED_AT DESC
        ) AS RN
    FROM YDS_DB.RAW.ARENAS_BRONZE
    WHERE TEAM_ABBR IS NOT NULL
      AND ARENA_NAME IS NOT NULL
      AND VALID_FROM IS NOT NULL
      AND VALID_TO IS NOT NULL
      AND VALID_FROM < VALID_TO
)
SELECT
    TEAM_ABBR,
    TRIM(UPPER(TEAM_NAME)) AS TEAM_NAME,
    TRIM(ARENA_NAME) AS ARENA_NAME,
    TRIM(CITY) AS CITY,
    VALID_FROM,
    VALID_TO,
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO
FROM dedup
WHERE RN = 1;

In [ ]:
%%sql -r arenas_validation
-- Validar: contagem e verificar join temporal funciona
SELECT COUNT(*) AS TOTAL_ARENAS,
    COUNT(DISTINCT TEAM_ABBR) AS TIMES_DISTINTOS
FROM YDS_DB.SILVER.ARENAS;

In [ ]:
%%sql -r arenas_test
-- Teste: arena do jogo dos 81 pontos via Silver
SELECT 
    g.DATE,
    g.ADVERSARIO_NOME,
    g.LOCAL_JOGO,
    g.KOBE_POINTS,
    CASE 
        WHEN g.LOCAL_JOGO = 'Home' THEN lal.ARENA_NAME
        ELSE opp.ARENA_NAME
    END AS ARENA,
    CASE 
        WHEN g.LOCAL_JOGO = 'Home' THEN lal.CITY
        ELSE opp.CITY
    END AS CIDADE
FROM YDS_DB.SILVER.KOBE_GAMELOG g
LEFT JOIN YDS_DB.SILVER.ARENAS lal 
    ON lal.TEAM_ABBR = 'LAL' AND g.DATE BETWEEN lal.VALID_FROM AND lal.VALID_TO
LEFT JOIN YDS_DB.SILVER.ARENAS opp 
    ON opp.TEAM_ABBR = g.ADVERSARIO_SIGLA AND g.DATE BETWEEN opp.VALID_FROM AND opp.VALID_TO
WHERE g.KOBE_POINTS >= 60
ORDER BY g.KOBE_POINTS DESC;